In [23]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict, List, Annotated
import operator

In [24]:
class SupportState(TypedDict):
    messages: Annotated[List[str], operator.add]
    user_id: str
    sentiment: str
    department: str
    next_step: str

In [25]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Triage

In [26]:
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import PromptTemplate

triage_prompt_template = PromptTemplate.from_template("""
You are a classifier. I give you the user query and you should route it into one of these classes:

- billing
- tech
- general

[user_history]
{history_json}
[/user_history]

[user_query]
{user_query}
[/user_query]

Return only the structured output.
""")

class TriageResponse(BaseModel):
    route: Literal["billing", "tech", "general"]
    
triage_llm = llm.with_structured_output(TriageResponse)

def triage_agent(state: SupportState):
    user_query = state["messages"][-1]
    history = state["messages"][:-1]
    
    prompt = triage_prompt_template.format(user_query=user_query, history_json=history)
    
    response = triage_llm.invoke(prompt)
    
    return {"department": response.route, "next_step": "route_to_department"}


def route_department(state: SupportState):
    return state["department"]

In [27]:
general_prompt_template = PromptTemplate.from_template("""
You are a general support agent. Answer simple product or account questions.
If you do not know the answer, say you do not know.

[user_history]
{history_json}
[/user_history]

[user_query]
{user_query}
[/user_query]

Return only the structured output.
""")

class GeneralResponse(BaseModel):
    answer: str
    
general_llm = llm.with_structured_output(GeneralResponse)

def general_agent(state: SupportState):
    user_query = state["messages"][-1]
    history = state["messages"][:-1]
    
    prompt = general_prompt_template.format(user_query=user_query, history_json=history)
    
    response = general_llm.invoke(prompt)
    
    return {"messages": [response.answer], "next_step": "done"}

In [28]:
billing_prompt_template = PromptTemplate.from_template("""
You are a billing support agent. Help with invoices, payments, refunds, plans, and subscriptions.
If you need private account details, ask the user for the missing information.

[user_history]
{history_json}
[/user_history]

[user_query]
{user_query}
[/user_query]

Return only the structured output.
""")

class BillingResponse(BaseModel):
    answer: str

billing_llm = llm.with_structured_output(BillingResponse)

def billing_agent(state: SupportState):
    user_query = state["messages"][-1]
    history = state["messages"][:-1]

    prompt = billing_prompt_template.format(user_query=user_query, history_json=history)

    response = billing_llm.invoke(prompt)

    return {"messages": [response.answer], "next_step": "done"}

In [29]:
tech_prompt_template = PromptTemplate.from_template("""
You are a technical support agent. Help with bugs, errors, setup, troubleshooting, and integrations.
Give clear steps and ask for logs or error messages when needed.

[user_history]
{history_json}
[/user_history]

[user_query]
{user_query}
[/user_query]

Return only the structured output.
""")

class TechResponse(BaseModel):
    answer: str

tech_llm = llm.with_structured_output(TechResponse)

def tech_agent(state: SupportState):
    user_query = state["messages"][-1]
    history = state["messages"][:-1]

    prompt = tech_prompt_template.format(user_query=user_query, history_json=history)

    response = tech_llm.invoke(prompt)

    return {"messages": [response.answer], "next_step": "done"}

In [30]:
builder = StateGraph(SupportState)

builder.add_node("triage", triage_agent)
builder.add_node("general", general_agent)
builder.add_node("billing", billing_agent)
builder.add_node("tech", tech_agent)

builder.add_edge(START, "triage")
builder.add_conditional_edges(
    "triage",
    route_department,
    {
        "general": "general",
        "billing": "billing",
        "tech": "tech",
    },
)

builder.add_edge("general", END)
builder.add_edge("billing", END)
builder.add_edge("tech", END)

support_graph = builder.compile()

In [31]:
initial_state = {
    "messages": ["I was charged twice this month. Can you help?"],
    "user_id": "user-123",
    "sentiment": "neutral",
    "department": "",
    "next_step": "",
}

support_graph.invoke(initial_state)

{'messages': ['I was charged twice this month. Can you help?',
  'I can help with that! To assist you further, could you please provide me with the following information:\n\n1. The email address associated with your account.\n2. The date of the charges you are referring to.\n3. Any transaction IDs or reference numbers if available.\n\nOnce I have this information, I can look into the duplicate charges for you.'],
 'user_id': 'user-123',
 'sentiment': 'neutral',
 'department': 'billing',
 'next_step': 'done'}